In [ ]:
# def get_closest_images(query_image_idx:int, num_results: int = 5)->list:
#     """
#     Retrieve the indices of the closest images to a given query image based on cosine distances.

#     Parameters:
#     - query_image_idx (int): Index of the query image in the feature vectors dataset.
#     - num_results (int): Number of closest images to retrieve (default is 5).

#     Returns:
#     - idx_closest (list): List of indices corresponding to the closest images to the query image.
#     """
#     distances = [ distance.cosine(pca_features[query_image_idx], feat) for feat in pca_features ]
#     idx_closest = sorted(range(len(distances)), key=lambda k: distances[k])[0:num_results+1]
#     similarity_score = 1 -  distance.cosine(features1, features2)
#     similarity_percentage = similarity_score * 100

#     return idx_closest

# def get_concatenated_images(images:list,indexes:list, thumb_height:int):
#     """
#     Concatenate a set of resized thumbnail images based on specified indices.

#     Parameters:
#     - images (list): List of image paths.
#     - indexes (list): List of indices indicating which images to concatenate.
#     - thumb_height (int): Height in pixels for resizing each thumbnail image.

#     Returns:
#     - concat_image (numpy.ndarray): Concatenated image array created from the specified thumbnails.
#     """
#     thumbs = []
#     for idx in indexes:
#         if idx!= len(images):
#           img = image.load_img(images[idx])
#           img = img.resize((int(img.width * thumb_height / img.height), thumb_height))
#           thumbs.append(img)
#     concat_image = np.concatenate([np.asarray(t) for t in thumbs], axis=1)
#     return concat_image

In [ ]:
import os
import keras
from keras.preprocessing import image
from keras.applications.imagenet_utils import decode_predictions, preprocess_input
from keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.spatial import distance

In [ ]:
model = keras.applications.VGG16(weights='imagenet', include_top=True)
#model.summary()

In [ ]:
def load_image(path:str):
    """
     A Function that formats the image to be processed using dnn.

     Parameters:
      - path = The image path of the image.

     Returns:
      - it return two value frist the new preprocess image and the input image data that can be fed into a neural network for prediction.
    """
    img = image.load_img(path, target_size=model.input_shape[1:3])
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    return img, x

# img, x = load_image("/content/heba.jpg")
# print("shape of x: ", x.shape)
# print("data type: ", x.dtype)
# plt.imshow(img)

In [ ]:
feat_extractor = Model(inputs=model.input, outputs=model.get_layer("fc2").output)

# feat_extractor.summary()

# img, x = load_image("/content/image.jpg")
# feat = feat_extractor.predict(x)

# plt.figure(figsize=(16,4))
# plt.plot(feat[0])

# for i, image_path in enumerate(images):
#   img, x = load_image(image_path)
#   feat = feat_extractor.predict(x)[0]
#   features.append(feat)

# images = [os.path.join(dp, f) for dp, dn, filenames in os.walk(images_path) for f in filenames if os.path.splitext(f)[1].lower() in image_extensions]

In [ ]:
images = []
np.save('images.npy', images)
features = []
np.save('features.npy', features)

In [ ]:
def get_names(images,indexes):
  images_names=[]
  for idx in indexes:
    if idx!= len(images):
      img=images[idx]
      base_name=os.path.basename(img)
      (file_Name,ext)=os.path.splitext(base_name)
      images_names.append(file_Name)
  return images_names


In [ ]:
def get_pathes(images, indexes):
    images_pathes = []
    for idx in indexes:
        if idx != len(images):
            img = images[idx]
            images_pathes.append(img)
    return images_pathes


In [ ]:
def show_res(test_img,res_image):
  plt.figure(figsize = (5,5))
  query_image=plt.imread(test_img)
  plt.imshow(query_image)
  plt.title("query image")
  plt.show()
  plt.imshow(res_image)
  plt.title("result images")

In [ ]:
def extract_features(image_path):

    img = image.load_img(image_path, target_size=(224, 224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    features = feat_extractor.predict(img)[0]
    features = features.flatten()
    return features

def image_similarity(image_path1, image_path2):

    features1 = extract_features(image_path1)
    features2 = extract_features(image_path2)

    similarity_score = 1 -  distance.cosine(features1, features2)

    similarity_percentage = similarity_score * 100

    return similarity_percentage

# image1_path = "/content/miza2.jpg"
# image2_path = "/content/images/id2.jpg"

# similarity_percentage = image_similarity(image1_path, image2_path)
# print(f"Similarity percentage between the two images: {int(similarity_percentage)}%")


In [ ]:
import copy


In [ ]:
def most_Similarity_images(folder_path,img_path,Number_Of_Reterned_Image):
  """
     A Function that search for an image that is close to the entered image in a specific folder.

     Parameters:
     - folder_path = The folder path you want to search in it.
     - img_path = The image path of the image.
     - Number_Of_Reterned_Image = number of images you want to return.

     Returns:
      - it Returns matching images that were selected using Number_Of_Reterned_Image.
  """
  images_path = folder_path
  image_extensions = ['.jpg', '.png', '.jpeg']
  max_num_images = 10000
  images = np.load('/content/images.npy', mmap_mode='r+').tolist()
  # print(len(images))
  new_images=[]
  for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if item_path not in images and os.path.splitext(item_path)[1].lower() in image_extensions:
      new_images.append(item_path)
  # print(len(new_images))
  if max_num_images < len(images):
      images = [images[i] for i in sorted(random.sample(range(len(images)), max_num_images))]

  features = np.load('features.npy', mmap_mode='r+').tolist()
  # print(len(features))

  new_features = []
  if len(new_images)>0:
    for i, image_path in enumerate(new_images):
      img, x = load_image(image_path)
      feat = feat_extractor.predict(x)[0]
      new_features.append(feat)
    images2=images+new_images
    np.save('/content/images.npy', images2)

    features2=features+new_features
    np.save('features.npy', features2)

    images = np.load('/content/images.npy')
    images=images.tolist()

    features = np.load('/content/features.npy')
    features=features.tolist()
  # print(len(features))



  img, x = load_image(img_path);
  feat = feat_extractor.predict(x)[0]
  features.append(feat)

  features = np.array(features)
  # pca = PCA(n_components=len(images))
  # pca.fit(features)
  # pca_features = pca.transform(features)


  similar_idx = [distance.cosine(features[len(features)-1], feat) for feat in features ]
  idx_closest = sorted(range(len(similar_idx)), key=lambda k: similar_idx[k])[0:Number_Of_Reterned_Image+1]

  results_image2=get_names(images,idx_closest)
  results_image2_path=get_pathes(images,idx_closest)

  similarity_percentage=[]
  for i in results_image2_path:
    percentage=image_similarity(img_path,i)
    similarity_percentage.append(percentage)
  final_res=[]
  final_res2=[]
  for i,x in zip(results_image2,similarity_percentage):
    final_res.append((i,x))

  if final_res[0][1]>58:
    final_res2.append(final_res[0])
  if final_res[1][1]>55:
    final_res2.append(final_res[1])
  if final_res[2][1]>48:
      final_res2.append(final_res[2])
  return final_res


im_path="/content/test.jpg"
results_image2=most_Similarity_images("/content/images",im_path,3)
print(results_image2)
# show_res(im_path,results_image)

In [ ]:
# def image_similarity(features1, features2):
#     similarity_score = 1 -  distance.cosine(features1, features2)
#     similarity_percentage = similarity_score * 100
#     return similarity_percentage

# # # Example usage
# # image1_path = "/content/miza2.jpg"
# # image2_path = "/content/images/id2.jpg"

# # similarity_percentage = image_similarity(image1_path, image2_path)
# # print(f"Similarity percentage between the two images: {int(similarity_percentage)}%")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from keras.preprocessing import image
from keras.applications.imagenet_utils import preprocess_input
from keras.applications.vgg19 import VGG19
from sklearn.cluster import KMeans

# Load pre-trained VGG19 model
model = VGG19(weights='imagenet', include_top=True)
feat_extractor = Model(inputs=model.input, outputs=model.get_layer("fc2").output)

# Define function to load and preprocess an image
def load_image(path):
    img = image.load_img(path, target_size=model.input_shape[1:3])
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    return img, x

# Extract features from images
def extract_features(image_folder):
    features = []
    image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith('.jpg')]
    for img_path in image_paths:
        img_features = feat_extractor.predict(load_image(img_path)[1])
        features.append(img_features.flatten())
    return features, image_paths

# Define image folder
image_folder = "/content/images"

# Extract features from images
image_features, image_paths = extract_features(image_folder)

# Apply K-means clustering
num_clusters = 4  # Adjust this value based on your requirement
kmeans = KMeans(n_clusters=num_clusters)
kmeans.fit(image_features)
cluster_labels = kmeans.labels_
kmeans.predict()
# Visualize clusters (Optional)
for cluster_id in range(num_clusters):
    print(f"Cluster {cluster_id}:")
    cluster_images = [image_paths[i] for i, label in enumerate(cluster_labels) if label == cluster_id]
    for img_path in cluster_images:
        img, _ = load_image(img_path)
        plt.imshow(img)
        plt.title(img_path)
        plt.show()
